# Семинар 1. Чтение данных из S3

Курсовой проект ML/DLOps · Inside Airbnb

В этом ноутбуке нужно заполнить **две функции**. Импорты, клиент S3 и примеры вызова уже готовы.

1. Сохраните `01_read_data.ipynb` в корне своего репозитория.
2. Подставьте bucket и key, которые выдаст преподаватель и положите копию того же CSV в `data/train.csv`.
3. Выполняйте ячейки сверху вниз. Сначала заполните TODO 1, затем TODO 2.

До заполнения функций `NotImplementedError` при их вызове — ожидаемый результат.

## Окружение и запуск

Команды выполняются **в терминале**, из папки репозитория:

```bash
conda create -n mlops26 python=3.11 -y
conda activate mlops26
conda install jupyterlab ipykernel pandas boto3 -y
python -m ipykernel install --user --name mlops26 --display-name "mlops26"
jupyter lab
```

Если окружение уже создано, повторять `conda create` не нужно.
В Jupyter откройте этот файл и выберите ядро **Python (mlops26)**.

In [ ]:
import sys
print("Python:", sys.executable)
print("Версия:", sys.version.split()[0])
# Путь должен указывать на окружение mlops26.

## Импорты и настройки

Подставьте bucket и key, которые выдаст преподаватель.
В клиенте замените `<ACCESS_KEY_ID>` и `<SECRET_ACCESS_KEY>` выданными значениями.
Перед отправкой ноутбука в Git верните плейсхолдеры вместо настоящих ключей и очистите выводы ячеек.

In [ ]:
from pathlib import Path

import boto3
import pandas as pd
from botocore.exceptions import BotoCoreError, ClientError
from secretkeys import ACCESS_KEY, KEY_ID
access_key=ACCESS_KEY
secret_key=KEY_ID
# Замените примеры значениями преподавателя.
S3_BUCKET = "mlops26-course-bucket"
S3_KEY = "prepared/train.csv"
LOCAL_PATH = Path("data/train.csv")

s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id= KEY_ID,
    aws_secret_access_key= ACCESS_KEY,
    region_name="ru-central1",
)

## TODO 1. Чтение CSV из S3

Заполните `read_s3_csv(bucket, key)`. Функция принимает имя bucket и полный ключ объекта и возвращает `DataFrame`.

Например, для `s3://course-bucket/prepared/listings.csv`:
- bucket: `course-bucket`;
- key: `prepared/listings.csv`.

После изменения определения функции выполните эту ячейку заново.

In [ ]:
def read_s3_csv(bucket, key):
    """Прочитайте CSV из S3 и верните DataFrame.

    1. Получите объект: s3.get_object(Bucket=bucket, Key=key).
    2. Откройте поток: with response["Body"] as body.
    3. Прочитайте CSV через pd.read_csv(body, dtype={"id": "string"}).
    4. Верните таблицу. Блок with закроет поток после чтения.
    """
    # Напишите реализацию вместо следующей строки.
    df = pd.read_csv(s3.get_object(Bucket=bucket,Key=key)["Body"])
    return df

### Первое чтение

Выполняйте после TODO 1. Проверьте первые строки, размер и названия колонок.
Если возникла ошибка S3, проверьте адрес и доступ к объекту.

In [ ]:
df_s3 = read_s3_csv(S3_BUCKET, S3_KEY)
display(df_s3.head())
print("Размер таблицы:", df_s3.shape)
print("Колонки:", df_s3.columns.tolist())

## TODO 2. Обработка ошибок и локальная копия

Заполните `read_data`. При доступном S3 возвращаем удалённые данные.
При сбое доступа или сети читаем локальный CSV, если путь передан.
Сообщение о переходе к файлу должно быть видно в выводе ячейки.

Ошибки формата CSV должны оставаться видимыми.

In [ ]:
def read_data(bucket, key, local_path=None):
    """Попробуйте S3, при сбое доступа или сети прочитайте локальную копию.

    1. В try вызовите read_s3_csv(bucket, key) и верните результат.
    2. Перехватите BotoCoreError, ClientError и OSError.
    3. Если local_path не передан, повторно поднимите исходную ошибку: raise.
    4. Иначе выведите тип ошибки и сообщение о переходе к локальному файлу.
    5. Прочитайте local_path через pd.read_csv с dtype={"id": "string"}.
       Верните DataFrame.

    Не используйте except Exception. Ошибки разбора CSV должны быть видны.
    Если локального файла нет, должен возникнуть FileNotFoundError.
    """
    try:
      df = read_s3_csv(bucket, key)
      return df
    except (BotoCoreError, ClientError, OSError) as exc:
       if local_path is None:
          raise
          
       else:
          print(f"{type(exc).__name__}: {exc}. "f"Переключаюсь на локальный файл: {local_path}")
          df = pd.read_csv(local_path, dtype={"id": "string"})
          return df


Немного не понял зачем тут лишний пробел после табуляции внутри функции

### Проверка: S3 доступен

Ожидаем таблицу из S3 без сообщения о переходе к локальному файлу.

In [ ]:
df = read_data(S3_BUCKET, S3_KEY, LOCAL_PATH) 
display(df.head())
print("Размер:", df.shape)

### Проверка: объект не найден

Намеренно меняем ключ. Ожидаем сообщение о переходе к `data/train.csv` и таблицу из локальной копии.
Перед запуском убедитесь, что локальный файл существует.

In [ ]:
LOCAL_PATH = 'data/train.csv' # (для теста)
df_local = read_data(S3_BUCKET, S3_KEY + ".missing", LOCAL_PATH)
display(df_local.head())
print("Размер:", df_local.shape)

### Дополнительная ручная проверка

Раскомментируйте **один** вызов за раз:

- нет объекта и не передан запасной путь — исходная ошибка S3;
- нет объекта и локального файла — `FileNotFoundError`.

Эти вызовы должны завершаться ошибкой. После проверки закомментируйте их обратно.

In [ ]:
# read_data(S3_BUCKET, S3_KEY + ".missing") +
# read_data(S3_BUCKET, S3_KEY + ".missing", "data/absent.csv") +

## Перед pull request

- Обе функции заполнены, обычное чтение и переход к файлу проверены.
- В README указаны conda-команды, источник данных и способ открытия ноутбука.
- В `.gitignore` добавлены `data/` и `.ipynb_checkpoints/`.
- Очистите выводы: **Kernel → Restart Kernel and Clear Outputs of All Cells**, затем сохраните файл.
- В коммит входят `01_read_data.ipynb`, README и `.gitignore`. CSV и ключи доступа остаются локально.

На следующих занятиях продолжаем этот же курсовой проект.